# MAC_WGS: Practical Exercises & Biological Exploration
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabloati/MAC_WGS/blob/main/upgrades/MAC_WGS_exercises_colab.ipynb)

**Course:** Microbial Assembly & Annotation using Whole Genome Sequencing (MAC_WGS)  
**Authors:** ConesaLab / University of Copenhagen Course Materials  

Welcome to the interactive exercise workbook! This notebook contains the three hands-on questionnaires designed to test both your **computational command-line skills** and your **biological critical thinking**.

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                                   WORKBOOK MODULES                                     │
├────────────────────────────────────────────────────────────────────────────────────────┤
│  Exercise 1: Raw Read Quality Control & Preprocessing (NanoPlot & NanoFilt)           │
│  Exercise 2: Assembly Dynamics, Graph Topology & Polishing (Flye, BUSCO, dnaapler)    │
│  Exercise 3: Hybrid Assembly & Technology Synthesis (SPAdes, Flye, Unicycler)         │
└────────────────────────────────────────────────────────────────────────────────────────┘
```


## ⚙️ Step 0: Environment & Workspace Setup

Run the cells below to initialize the Conda/Mamba environment and download the course datasets and pre-computed results.


In [ ]:
# 1. Install condacolab
!pip install -q condacolab
import condacolab
condacolab.install()


In [ ]:
# 2. Install bioinformatics tools
!mamba install -c conda-forge -c bioconda nanoplot nanofilt flye busco minimap2 racon dnaapler prokka bandage unicycler samtools -y


In [ ]:
# 3. Setup folders and download course deliverables (single archive with ERDA fallback)
!mkdir -p upgrades results
!wget -q --show-progress https://github.com/pabloati/MAC_WGS/releases/download/v1.0-data/mac_wgs_data.tar.gz || \
 (echo "--> Downloading from ERDA backup..." && \
  mkdir -p data/bsubtilis data/ecoli && \
  wget -q -O data/bsubtilis/bsubtilis_long_reads.fastq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/bsubtilis/bsubtilis_long_reads.fastq && \
  wget -q -O data/ecoli/illumina_f.fq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/ecoli/illumina_f.fq && \
  wget -q -O data/ecoli/illumina_r.fq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/ecoli/illumina_r.fq && \
  wget -q -O data/ecoli/minion_2d.fq https://sid.erda.dk/share_redirect/ePr2eWTdSX/data/ecoli/minion_2d.fq)

# Extract dataset archive if downloaded
![ -f mac_wgs_data.tar.gz ] && tar -xzf mac_wgs_data.tar.gz

# Extract pre-computed deliverables archive to populate results/
!wget -q -O upgrades/precomputed_results.tar.gz https://raw.githubusercontent.com/pabloati/MAC_WGS/main/upgrades/precomputed_results.tar.gz || true
!tar -xvf upgrades/precomputed_results.tar.gz -C .

# Validate files
!ls -lh data/bsubtilis data/ecoli
print("Environment and results workspace initialized successfully!")


---

# 📝 Exercise 1: Raw Read Quality Control & Preprocessing

**Dataset:** *Bacillus subtilis* Oxford Nanopore Technologies (ONT) Long Reads  
**Target Files:** `results/bsubtilis/qc/bsubtilis_raw_NanoStats.txt`, `data/bsubtilis/bsubtilis_long_reads.fastq`, `results/bsubtilis/filtered/bsubtilis_filtered.fastq`

---

### Question 1: Sequenced Yield & Theoretical Coverage
Sequencing coverage (depth) determines whether an assembler has enough overlapping evidence to reconstruct a genome without gaps.

* **Task:** Using bash commands (`grep`, `awk`, or `cat`), extract the total number of sequenced bases from `results/bsubtilis/qc/bsubtilis_raw_NanoStats.txt` and calculate the theoretical sequencing depth assuming a *Bacillus subtilis* genome size of $4.2	ext{ Mb}$ ($4,200,000	ext{ bp}$).


In [ ]:
# [Q1 Code] Write your bash / python one-liner below:
!


* **Q1 Critical Thinking:**
1. What theoretical coverage did you calculate?
   * *Your Answer:*
2. According to Schatz's long-read sequencing coverage guidelines ("The Chef's Recipe": $\ge 20	ext{--}30	imes$ for ONT), is this dataset sufficient for *de novo* assembly without short-read assistance?
   * *Your Answer:*


---

### Question 2: Read Length Metrics — Mean vs. Read N50
In long-read sequencing, read length distributions are highly skewed (many moderate-length reads and a long tail of ultra-long reads).

* **Task:** Query `results/bsubtilis/qc/bsubtilis_raw_NanoStats.txt` to find the **Mean Read Length**, the **Read Length N50**, and the **Longest Read**.


In [ ]:
# [Q2 Code] Write your command below:
!


* **Q2 Critical Thinking:**
1. Define **Read N50** in your own words. Why is Read N50 ($7,561	ext{ bp}$) substantially larger than the Mean Read Length ($5,414	ext{ bp}$)?
   * *Your Answer:*
2. The longest read in this dataset is $85,025	ext{ bp}$. Knowing that the longest repetitive element in *B. subtilis* is the ribosomal RNA operon ($rrn$, ~5,000–7,000 bp), what does this tell you about the ability of this dataset to bridge bacterial repeats?
   * *Your Answer:*


---

### Question 3: Phred Quality Scores & Expected Error Rates
Nanopore basecallers assign a Phred quality score ($Q$) to each base, defined on a logarithmic scale:
$$Q = -10 \log_{10}(P_{	ext{error}}) \iff P_{	ext{error}} = 10^{-Q/10}$$

* **Task:** Query `results/bsubtilis/qc/bsubtilis_raw_NanoStats.txt` to identify the percentage of sequenced bases above $Q10$, $Q15$, and $Q20$.


In [ ]:
# [Q3 Code] Write your command below:
!


* **Q3 Critical Thinking:**
1. The mean read quality of this run is $Q16.3$. What is the base-level error probability ($P_{	ext{error}}$) corresponding to $Q16.3$?
   * *Your Answer:*
2. How many basecalling errors are expected for every 1,000 base pairs sequenced at this quality?
   * *Your Answer:*
3. If $82.6\%$ of the yield is $>Q15$ and only $7.5\%$ is $>Q20$, what does this indicate about the chemistry generation (R9.4.1 vs R10.4.4 duplex)?
   * *Your Answer:*


---

### Question 4: Filtering Dynamics & Assembly Noise Reduction
Short reads act as graph-tangling noise during long-read assembly without providing bridging benefits. In Step 1, reads were filtered with `NanoFilt -l 1000 -q 12`.

* **Task:** Write a bash command to count how many reads existed in the raw dataset (`data/bsubtilis/bsubtilis_long_reads.fastq`) versus how many reads survived filtering (`results/bsubtilis/filtered/bsubtilis_filtered.fastq`).


In [ ]:
# [Q4 Code] Write your command below:
!


* **Q4 Critical Thinking:**
1. What percentage of reads was discarded during filtering?
   * *Your Answer:*
2. Why is discarding short ($< 1,000	ext{ bp}$) reads beneficial for Flye/OLC graph assemblers in microbial genomics, whereas in Illumina assembly discarding 150 bp reads would cause severe coverage loss?
   * *Your Answer:*


---

### Question 5: Quality Profile Diagnostics (NanoPlot Inspection)
Inspect the generated NanoPlot report and quality statistics.


In [ ]:
# [Q5 Code] Run a command to display key summary stats from NanoStats.txt:
!cat results/bsubtilis/qc/bsubtilis_raw_NanoStats.txt | head -n 15


* **Q5 Critical Thinking & Diagnostics:**
1. In a healthy long-read run, what shape should the 2D density cluster (Read Length vs Quality) exhibit?
   * *Your Answer:*
2. If a sequencing run displays a prominent secondary cluster of high-abundance reads concentrated strictly at $\sim 1.5	ext{ kb}$, what common biological or library-preparation artifact could that represent in a microbiology laboratory?
   * *Your Answer:*
3. What would a cluster of extremely short reads ($< 200	ext{ bp}$) with low quality scores indicate regarding pre-sequencing sample prep (e.g. DNA shearing / fragmented extraction)?
   * *Your Answer:*


---

# 🧬 Exercise 2: Assembly Dynamics, Graph Topology & Polishing

**Dataset:** *Bacillus subtilis* Long-Read Assembly & Polishing  
**Target Files:** `results/bsubtilis/assembly/assembly_info.txt`, `results/bsubtilis/assembly/assembly_graph.gfa`, `results/bsubtilis/busco_flye/`, `results/bsubtilis/busco_polished/`, `results/bsubtilis/final_assembly.fasta`

---

### Question 1: Dissecting Assembly Contigs & Replicons
`Flye` produces an `assembly_info.txt` file summarizing the status of each reconstructed contig.

* **Task:** Display and inspect the contents of `results/bsubtilis/assembly/assembly_info.txt`.


In [ ]:
# [Q1 Code] Write your command below:
!cat results/bsubtilis/assembly/assembly_info.txt


* **Q1 Critical Thinking:**
1. Identify which contig corresponds to the bacterial **chromosome** and which contig corresponds to a **plasmid**. What are their respective lengths?
   * *Your Answer:*
2. Notice the `cov.` (coverage depth) column: `contig_1` has $60	imes$ depth, while `contig_4` has $109	imes$ depth. What biological reason explains why the plasmid has nearly **double the sequencing depth** of the main chromosome?
   * *Your Answer:*
3. Look at the `circ.` column. What does `Y` indicate, and how does Flye confirm that a contig is physically circular?
   * *Your Answer:*


---

### Question 2: Graph Topology & Repeat Resolution in Bandage
Visualizing assembly graphs (`.gfa`) is essential to diagnose whether an assembly is complete or fragmented.

* **Task:** Render `results/bsubtilis/assembly/assembly_graph.gfa` to PNG using Bandage CLI and display it inline in Colab.


In [ ]:
# [Q2 Code] Render and display the assembly graph:
!Bandage image results/bsubtilis/assembly/assembly_graph.gfa bsubtilis_graph.png --height 500

from IPython.display import Image
Image('bsubtilis_graph.png')


* **Q2 Critical Thinking:**
1. Describe the topology of the assembly graph. Do you see a single closed circular loop, or multiple branching nodes?
   * *Your Answer:*
2. The fundamental theorem of genome assembly states that repeats are resolved when:
   $$	ext{Read Length } (L) > 	ext{Repeat Length } (A)$$
   Explain why Oxford Nanopore reads assembled with Flye produced a single circular chromosome, whereas short 150 bp Illumina reads typically produce 50–150 fragmented contigs in *Bacillus subtilis*.
   * *Your Answer:*


---

### Question 3: Benchmarking Gene Completeness with BUSCO
`BUSCO` assesses the presence of evolutionarily conserved single-copy orthologous genes. For *B. subtilis*, we use the `bacillales_odb10` lineage database (450 total benchmark genes).

* **Task:** Write a bash command to parse the BUSCO summary reports for both the **raw Flye assembly** and the **Racon polished assembly**.


In [ ]:
# [Q3 Code] Parse BUSCO summary reports:
!


* **Q3 Critical Thinking:**
1. Record the Complete ($C$), Single-copy ($S$), Duplicated ($D$), Fragmented ($F$), and Missing ($M$) percentages for both assemblies.
   * *Raw Flye:*
   * *Racon Polished:*
2. Did the raw long-read Flye assembly achieve high completeness prior to any polishing? What does this demonstrate about the consensus quality of Flye's internal graph consensus algorithm?
   * *Your Answer:*


---

### Question 4: The Polishing Paradox & Homopolymer Indels
When comparing the BUSCO scores, you will observe that BUSCO completeness slightly dropped from $99.8\%$ ($449/450$ complete) to $99.3\%$ ($447/450$ complete) after Racon polishing with long reads.

* **Q4 Critical Thinking & Biochemical Reasoning:**
1. Why did BUSCO completeness **drop** after running Racon polishing with the same long-read dataset?
   * *Your Answer:*
2. **The Homopolymer Mechanism:** Oxford Nanopore sequencing measures ionic current as DNA passes through a biological nanopore. When the pore encounters a homopolymer stretch (e.g., `AAAAAAA`), the ionic current remains flat, making it difficult to determine exact base counts. Explain how 1-bp insertion or deletion (indel) errors introduced during over-polishing cause **frameshifts** in protein-coding genes, causing BUSCO to misclassify intact genes as "Fragmented".
   * *Your Answer:*
3. When would polishing with Racon be **beneficial** versus when should you skip it?
   * *Your Answer:*


---

### Question 5: Standardizing Chromosome Synteny (`dnaapler`)
Bacterial chromosomes are circular, meaning an assembler can arbitrarily choose any base as "coordinate 1".

* **Task:** Write a command to compare the sequence start of the raw Flye assembly (`results/bsubtilis/assembly/assembly.fasta`) against the `dnaapler` reoriented assembly (`results/bsubtilis/final_assembly.fasta`).


In [ ]:
# [Q5 Code] Write your command below:
!


* **Q5 Critical Thinking:**
1. What biological feature is located at the origin of replication ($oriC$) in most bacteria?
   * *Your Answer:*
2. Why is running `dnaapler` to standardize coordinate 1 at the *dnaA* start codon a mandatory prerequisite before depositing genomes into NCBI GenBank or comparing synteny between different isolates?
   * *Your Answer:*


---

# ⚔️ Exercise 3: Hybrid Assembly & Technology Synthesis

**Dataset:** *Escherichia coli* C-1 Multi-Platform Benchmark (Illumina PE + ONT MinION)  
**Target Files:** `results/ecoli/unicycler_ecoli/001_spades_graph_k127.gfa`, `results/ecoli/unicycler_ecoli/assembly.gfa`, `results/ecoli/unicycler_ecoli/unicycler.log`, `results/ecoli/flye_ecoli/assembly_info.txt`

---

### Question 1: The Short-Read Repeat Knot (SPAdes Graph Inspection)
Short reads ($150	ext{ bp}$) possess high base accuracy ($>99.9\%$) but cannot bridge repetitive regions longer than the library insert size.

* **Task:** Render the initial short-read SPAdes graph from Unicycler (`001_spades_graph_k127.gfa`) using Bandage CLI and count the number of contig segments in this graph.


In [ ]:
# [Q1 Code] Render SPAdes short-read graph and count segments:
!Bandage image results/ecoli/unicycler_ecoli/001_spades_graph_k127.gfa ecoli_spades_graph.png --height 500

from IPython.display import Image
Image('ecoli_spades_graph.png')


In [ ]:
# [Q1 Code] Count segments in the SPAdes graph:
!


* **Q1 Critical Thinking:**
1. Inspect the graph image. Why does the short-read assembly form a complex tangled "hairball" or knot rather than a clean circular ring?
   * *Your Answer:*
2. Notice the high-coverage hub nodes that have multiple incoming and outgoing branch connections. In *E. coli*, there are **7 identical copies of the 5.5 kb ribosomal RNA ($rrn$) operon**. Why is it mathematically impossible for $150	ext{ bp}$ paired-end reads to resolve which upstream gene connects to which downstream gene across these operons?
   * *Your Answer:*


---

### Question 2: The Long-Read Bridging Mechanism in Unicycler
`Unicycler` uses long reads as "bridges" to find the correct biological path through the tangled short-read de Bruijn graph.

* **Task:** Write a bash script to track the reduction in graph segment count across Unicycler's execution stages (`001_spades_graph_k127.gfa` $ightarrow$ `005_bridges_applied.gfa` $ightarrow$ `assembly.gfa`), and query `unicycler.log` for bridge resolution statistics.


In [ ]:
# [Q2 Code] Track segment count reduction:
!


* **Q2 Critical Thinking:**
1. How does Unicycler align long ONT reads across the $rrn$ repeat nodes to determine the true contiguous path?
   * *Your Answer:*
2. Why does combining short reads with long reads yield a **single circular chromosome ($N=1$)** without the risk of structural misassemblies?
   * *Your Answer:*


---

### Question 3: Contiguity vs. Depth in Long-Read Assembly (Flye)
In Part 1, `Flye` assembled *B. subtilis* into 1 circular chromosome. However, when running Flye on *E. coli* C-1 (`minion_2d.fq`), the assembly resulted in **18 contigs** ($N50 = 608	ext{ kb}$).

* **Task:** Query `results/ecoli/flye_ecoli/assembly_info.txt` to extract the contig lengths, repeat classifications, and average sequencing depth.


In [ ]:
# [Q3 Code] Write your command below:
!


* **Q3 Critical Thinking:**
1. Look at the `cov.` column. What was the average sequencing depth for *E. coli* ($16	ext{--}18	imes$) compared to *B. subtilis* ($60	imes$)?
   * *Your Answer:*
2. The *E. coli* dataset was sequenced using early 2D MinION chemistry with an error rate of $\sim 18\%$. Why did the low coverage ($16	imes$) and high error rate prevent Flye from completely circularizing the chromosome into 1 contig, while Unicycler achieved complete circularization?
   * *Your Answer:*
3. What is the minimum recommended coverage for Oxford Nanopore reads when attempting *de novo* assembly without short-read polishing?
   * *Your Answer:*


---

### Question 4: Biological Accuracy & The "CDS Inflation" Paradox
Sequencing accuracy directly impacts functional annotation. A single basecall indel in an unpolished long-read assembly alters the reading frame.

* **Task:** Write a command to compare the total predicted coding sequences (CDS) between the **unpolished Flye assembly** (~4,650 CDS) and the **Unicycler hybrid assembly** (~4,280 CDS).


In [ ]:
# [Q4 Code] Compare predicted CDS counts:
!


* **Q4 Critical Thinking & Biological Reasoning:**
1. *E. coli* C-1 has a known standard genome containing $\sim 4,200	ext{--}4,300$ functional protein-coding genes. Why does the unpolished Flye assembly predict **$\sim 370$ extra genes**?
   * *Your Answer:*
2. Explain the biochemical mechanism: How does a 1-bp deletion in a 1,200 bp coding sequence introduce a premature stop codon, causing `Prodigal` to predict two truncated 600 bp "pseudogenes" instead of one intact enzyme?
   * *Your Answer:*
3. Why is an inflated CDS count in a draft long-read assembly a major **warning sign** of uncorrected homopolymer errors rather than evidence of novel biological functions?
   * *Your Answer:*


---

### Question 5: Decision Matrix for Real-World Microbiology
As a bioinformatician, you must select the most cost-effective and scientifically appropriate sequencing strategy based on project goals.

* **Q5 Critical Thinking & Scenario Analysis:**
  For each of the following three research scenarios, select the optimal strategy (**Short-Read Only**, **Long-Read Only**, or **Hybrid**) and provide a clear 2-sentence biological justification:

  * **Scenario A (Clinical Outbreak Tracking):**  
    *Goal:* Track an ongoing hospital outbreak of Carbapenem-Resistant *Klebsiella pneumoniae* across 50 patient isolates over a weekend on a strict budget. You need high-resolution Single Nucleotide Polymorphism (SNP) phylogenetic trees.
    * *Your Choice & Justification:*

  * **Scenario B (AMR Plasmid & Pathogenicity Island Characterization):**  
    *Goal:* Determine whether a newly discovered colistin resistance gene ($mcr	ext{-}1$) is integrated into the host chromosome or located on a transmissible 90 kb conjugative plasmid flanked by insertion sequences.
    * *Your Choice & Justification:*

  * **Scenario C (Official Species Type Strain Submission):**  
    *Goal:* Generate a definitive, publication-quality reference genome for a newly discovered extremophilic bacterium to deposit in the DSMZ culture collection and NCBI RefSeq.
    * *Your Choice & Justification:*
